In [1]:
!pip install -q pymongo


**IMPORT**

In [2]:
from google.colab import userdata   # lit les secrets Colab (🔑)
from pymongo import MongoClient

In [5]:
uri = userdata.get("MONGO_URI")
print("Longueur :", len(uri))
print("Début :", repr(uri[:30]))
print("Fin :", repr(uri[-31:]))

Longueur : 79
Début : 'mongodb+srv://hinataa92i_db_us'
Fin : '2@cluster0.m5qskyq.mongodb.net/'


In [6]:
client = MongoClient(userdata.get("MONGO_URI"))
db = client["cinema"]

print(client.admin.command("ping"))                       # {'ok': 1} = connecté
print("Films bruts :", db["films_bruts"].count_documents({}))

{'ok': 1}
Films bruts : 6015


**Etat des donnés AVANT nettoyage**

In [7]:
brut = db["films_bruts"]
print("Budget à 0 :", brut.count_documents({"budget": 0}))
print("Recettes à 0 :", brut.count_documents({"revenue": 0}))
print("Durée à 0 :", brut.count_documents({"runtime": 0}))

Budget à 0 : 1526
Recettes à 0 : 1204
Durée à 0 : 1


**PIPELINE DE NETTOYAGE**

In [8]:
pipeline_nettoyage = [
    # Étape 1 : sélection, renommage et nettoyage des champs
    {"$project": {
        "_id": 1,
        "titre": "$title",
        "titre_original": "$original_title",
        "langue_originale": "$original_language",
        "date_sortie": {"$dateFromString": {"dateString": "$release_date", "onError": None}},
        "duree_min": {"$cond": [{"$gt": ["$runtime", 0]}, "$runtime", None]},
        "budget":    {"$cond": [{"$gt": ["$budget", 0]}, "$budget", None]},
        "recettes":  {"$cond": [{"$gt": ["$revenue", 0]}, "$revenue", None]},
        "popularite": "$popularity",
        "note_moyenne": "$vote_average",
        "nb_votes": "$vote_count",
        "genres": "$genres.name",                         # liste d'objets → liste de noms
        "pays_production": "$production_countries.iso_3166_1",
        "societes_production": "$production_companies.name",
        "saga": "$belongs_to_collection.name",
    }},
    # Étape 2 : ajout de l'année de sortie
    {"$set": {"annee": {"$year": "$date_sortie"}}},
    # Étape 3 : écriture dans la collection "films"
    {"$out": "films"},
]

list(brut.aggregate(pipeline_nettoyage))   # exécute le pipeline
print("Collection 'films' créée :", db["films"].count_documents({}), "documents")

Collection 'films' créée : 6015 documents


**Vérification des données APRES nettoyage**

In [9]:
films = db["films"]
print("Budget nul (inconnu) :", films.count_documents({"budget": None}))
print("Recettes nulles (inconnues) :", films.count_documents({"recettes": None}))
print("Budget à 0 restant :", films.count_documents({"budget": 0}))   # doit être 0

films.find_one({"titre": "Inception"})

Budget nul (inconnu) : 1526
Recettes nulles (inconnues) : 1204
Budget à 0 restant : 0


{'_id': 27205,
 'titre': 'Inception',
 'titre_original': 'Inception',
 'langue_originale': 'en',
 'date_sortie': datetime.datetime(2010, 7, 15, 0, 0),
 'duree_min': 148,
 'budget': 160000000,
 'recettes': 839030630,
 'popularite': 52.8317,
 'note_moyenne': 8.373,
 'nb_votes': 40808,
 'genres': ['Action', 'Science-Fiction', 'Crime'],
 'pays_production': ['GB', 'US'],
 'societes_production': ['Warner Bros. Pictures',
  'Legendary Pictures',
  'Syncopy'],
 'annee': 2010}

On souhaite obtenir des fiches de films propres et fiables.

On remplace les 0 par "inconnu", on convertit les dates, et simplifie les listes (genres, pays, sociétés) et on retire toutes les données qui ne seront pas utiles pour le projet.


On obtient une collection de films, avec un document par films, uniquement des informations utiles et claires pour la suite de nos analyses.

**PIPELINE 2 : Chaque film contient une liste d'acteurs et membres d'équipes de tournage. Objectif => crééer 1 personne dans un film**

In [10]:
pipeline_credits = [
    # Étape 1 : une seule liste "personnes" avec acteurs + équipe, au même format
    {"$project": {
        "personnes": {"$concatArrays": [
            {"$map": {"input": "$credits.cast", "as": "a", "in": {   # $$a = un acteur
                "personne_id": "$$a.id", "nom": "$$a.name",
                "type": "acteur", "role": "$$a.character", "ordre": "$$a.order"}}},
            {"$map": {"input": "$credits.crew", "as": "m", "in": {   # $$m = un membre d'équipe
                "personne_id": "$$m.id", "nom": "$$m.name",
                "type": "equipe", "role": "$$m.job"}}},
        ]}
    }},
    # Étape 2 : un document par personne
    {"$unwind": "$personnes"},
    # Étape 3 : mise en forme finale
    {"$project": {
        "_id": 0,                       # MongoDB générera un nouvel identifiant
        "film_id": "$_id",
        "personne_id": "$personnes.personne_id",
        "nom": "$personnes.nom",
        "type": "$personnes.type",
        "role": "$personnes.role",
        "ordre": "$personnes.ordre",
    }},
    # Étape 4 : écriture
    {"$out": "credits"},
]

list(brut.aggregate(pipeline_credits))
print("Crédits créés :", db["credits"].count_documents({}))

Crédits créés : 102121


L'objectif est de pouvoir répondre facilement aux questions "qui a travaillé sur quoi ? " .

On ressort les équipes de tournages ainsi que les acteurs des films, et on créé une ligne par participation.
"Brad Pitt a joué Tyler Durden dans Fight Club" => 1 ligne

On obtient alors la collection "crédits" avec 102 121 participations.

Cela permet de faire par la suite des jointures entre les participants et les films.


**PIPELINE 3 : 1 personne apparaît dans plusieurs films. On veut regrouper les films pour 1 personnes**

In [11]:
pipeline_personnes = [
    {"$group": {
        "_id": "$personne_id",
        "nom": {"$first": "$nom"},
        "films": {"$addToSet": "$film_id"},   # liste des films, sans doublon
        "types": {"$addToSet": "$type"},      # acteur, equipe ou les deux
    }},
    {"$project": {"nom": 1, "types": 1, "nb_films": {"$size": "$films"}}},
    {"$out": "personnes"},
]

list(db["credits"].aggregate(pipeline_personnes))
print("Personnes créées :", db["personnes"].count_documents({}))

# Les 5 personnes présentes dans le plus de films
for p in db["personnes"].find().sort("nb_films", -1).limit(5):
    print(p["nom"], "→", p["nb_films"], "films", p["types"])

Personnes créées : 38865
Christophe Beck → 88 films ['equipe']
Marco Beltrami → 87 films ['equipe']
Jason Blum → 86 films ['equipe']
Hans Zimmer → 80 films ['equipe']
John Debney → 73 films ['equipe']


On souhaite que chaque personne existe 1 seule fois et 1 seule ligne avec ses informations.

Hans Zimmer apparait dans 80 films donc 80 lignes, on veut une seule fiche Hans Zimmer indiquant qu'il a participé à 80 films en tant que membre d'équipe.

Création d'index sur film_id et personne_id dans credits : comme l'index d'un livre, ils permettent à MongoDB de retrouver rapidement les crédits d'un film ou d'une personne, ce qui accélérera les jointures ($lookup) des prochains pipelines.

In [12]:
db["credits"].create_index("film_id")
db["credits"].create_index("personne_id")
print("Index créés")

Index créés
